# 回帰分析（演習）
## パッケージ内データと外部データ読み込み

分析の最初の関門は、データをどこからどう手に入れるかである。教科書の例題として配られることもあれば、役所が公表した統計を自分で整理して使うこともある。入手の仕方が違えば、読み込み方も、分析の前にすべき確認事項も変わる。

## 0. 今回の問い

市場価値の大きい企業ほど、投資額も大きいのか。また、人口密度の高い地域ほど、住宅の床面積は小さいのか。

前半では、`statsmodels` に同梱されたGrunfeld企業投資データから米国企業11社の1935年と1954年を取り出し、市場価値と投資額の関係を単回帰でとらえる。あわせて、同じ関係が年によってどう変わるかを見る。  
後半では、公的統計を整理した都道府県別の人口密度と住宅面積の二つのCSVを都道府県コードで結合し、密度の高い地域で住宅面積がどうなるかを調べる。

前者はパッケージに同梱されたデータ、後者は複数の外部CSVである。値そのものだけでなく、どのように読み込み、何を確認してから分析に入るかも、今回の主題である。

In [ ]:
%pip install japanize-matplotlib-jlite -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import japanize_matplotlib_jlite

## 1. パッケージ内のデータを読み込む

`statsmodels` には、統計分析の例題や動作確認に使えるデータが同梱されている。パッケージ内データは、ファイルの場所を指定せず、決められた関数を呼び出して読み込める。

ここでは、Grunfeld企業投資データを使う。このデータには、米国企業11社について、1935年から1954年までの投資額、市場価値、資本ストックが収録されている。

データ全体は、同じ企業を複数年観測したパネルデータである。ここでは特定の年だけを取り出し、11企業からなる横断面データとして単回帰を行う。

## 2. データ本体と説明書を確認する

データを使う前に、値だけでなく説明書を読む。`NOTE` には観測数、変数の数、列の意味、単位、対象期間が記録されている。

In [ ]:
print(sm.datasets.grunfeld.NOTE)

In [ ]:
grunfeld = sm.datasets.grunfeld.load_pandas().data.copy()

grunfeld["year"] = grunfeld["year"].astype(int)

print(grunfeld.shape)
grunfeld.head()

Grunfeldデータの主な変数は次のとおりである。

| 列名 | 内容 |
|---|---|
| `invest` | 総投資額|
| `value` | 12月31日時点の企業の市場価値|
| `capital` | 工場・設備の資本ストック|
| `firm` | 企業名 |
| `year` | 年、1935年から1954年 |

この分析で使う変数は、`invest` と `value` の二つである。`capital` はデータに含まれているが、今回の回帰式には使用しない。

## 3. データ全体の構造を確認する

このデータでは、同じ企業が複数年にわたって観測されている。

- 横断面データでは、通常、1行が1企業である。
- 時系列データでは、通常、1行が1時点である。
- このデータでは、1行が「企業と年の組合せ」である。

企業数、年数、企業ごとの観測数を確認する。企業ごとの観測数は、`groupby("firm")` で企業ごとにまとめ、`size()` で各グループの行数を数えて求める。

In [ ]:
print("企業数:", grunfeld["firm"].nunique())
print("最初の年:", grunfeld["year"].min())
print("最後の年:", grunfeld["year"].max())
print("年数:", grunfeld["year"].nunique())
print("企業・年の重複:", grunfeld.duplicated(["firm", "year"]).sum())

grunfeld.groupby("firm").size().rename("観測数")

## 4. 1954年の横断面データを作る

同じ企業を繰り返し含めないよう、最終年である1954年だけを取り出す。

これにより、1行が1企業となる横断面データになる。観測数は11と少ないため、推定結果は少数の大企業に影響されやすい。この点は、結果を解釈する際の重要な注意事項である。

In [ ]:
grunfeld_1954 = (
    grunfeld[grunfeld["year"] == 1954]
    .copy()
    .sort_values("value")
    .reset_index(drop=True)
)

print("観測数:", len(grunfeld_1954))
grunfeld_1954[["firm", "invest", "value"]]

## 5. 基本統計量と散布図

中心となる問いは、次のとおりである。

> 1954年の企業間比較では、市場価値が大きい企業ほど投資額も大きいか。

市場価値と投資額の基本統計量、相関係数、散布図を確認する。

In [ ]:
print(grunfeld_1954[["invest", "value"]].describe().T)

display(
    grunfeld_1954[["invest", "value"]]
    .describe()
    .T
)

print(
    "相関係数:",
    grunfeld_1954["invest"].corr(grunfeld_1954["value"]),
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    grunfeld_1954["value"],
    grunfeld_1954["invest"],
)

# iterrows() はデータを1行ずつ取り出す。ここでは企業名を各点の隣に書き添える。
for _, row in grunfeld_1954.iterrows():
    plt.annotate(
        row["firm"],
        (row["value"], row["invest"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8,
    )

plt.title("1954年の企業市場価値と投資額")
plt.xlabel("市場価値")
plt.ylabel("投資額")
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

### 演習1　散布図から仮説を立てる

散布図と相関係数を見て、次を説明しなさい。

1. 市場価値と投資額には、正、負、無相関のどれが見られるか。
2. 関係はおおむね直線的に見えるか。
3. 他の企業から大きく離れた企業はどこか。
4. 観測数が11しかないことは、回帰結果の安定性にどのような影響を与えるか。
5. 散布図だけから、市場価値の上昇が投資を増加させるという因果関係を結論づけられるか。

In [ ]:
# ここに説明をコメントとして入力する


## 6. 1954年の単回帰

投資額を市場価値だけで説明する。

$$
invest_i
=
\beta_0
+
\beta_1 value_i
+
u_i
$$

$i$ は企業を表す。

- $\beta_0$ は、市場価値が0の企業の投資額の推定値である。ただし、市場価値0は観測範囲外なので、実質的な意味は限定的である。
- $\beta_1$ は、市場価値が1単位大きい企業では、投資額が平均的に何単位大きいかを表す。

In [ ]:
Y_1954 = grunfeld_1954["invest"]
X_1954 = sm.add_constant(grunfeld_1954[["value"]])

model_1954 = sm.OLS(Y_1954, X_1954)
result_1954 = model_1954.fit()

result_1954.summary()

## 7. 推定値と不確実性を確認する

回帰表全体には多くの情報がある。ここでは、係数、標準誤差、t値、p値、95％信頼区間、決定係数に注目する。

帰無仮説は

$$
H_0:\beta_1=0
$$

である。これは、1954年の企業間比較で、市場価値と投資額に線形な関係がないという仮説である。

In [ ]:
result_table_1954 = pd.DataFrame({
    "推定値": result_1954.params,
    "標準誤差": result_1954.bse,
    "t値": result_1954.tvalues,
    "p値": result_1954.pvalues,
})

conf_1954 = result_1954.conf_int(alpha=0.05)
conf_1954.columns = ["95%信頼区間 下限", "95%信頼区間 上限"]

display(result_table_1954.join(conf_1954))
print("決定係数 R^2:", result_1954.rsquared)

### 演習2　単回帰を解釈する

出力を使って、次の文章を完成させなさい。

- 市場価値が1単位大きい企業では、投資額は平均約（　　　）単位大きい。
- 市場価値の係数のp値は（　　　）である。
- 係数の95％信頼区間は、およそ（　　　）から（　　　）である。
- 決定係数は約（　　　）である。

## 8. 回帰直線を重ねる

当てはめ値は

$$
\widehat{invest}_i
=
\widehat{\beta}_0
+
\widehat{\beta}_1 value_i
$$

である。

散布図に回帰直線を重ね、各企業が直線からどの程度離れているかを確認する。

In [ ]:
grunfeld_1954["fitted"] = result_1954.fittedvalues
grunfeld_1954["residual"] = result_1954.resid

line_data_1954 = grunfeld_1954.sort_values("value")

plt.figure(figsize=(8, 5))
plt.scatter(
    grunfeld_1954["value"],
    grunfeld_1954["invest"],
    label="実際の企業",
)
plt.plot(
    line_data_1954["value"],
    line_data_1954["fitted"],
    label="回帰直線",
)

for _, row in grunfeld_1954.iterrows():
    plt.annotate(
        row["firm"],
        (row["value"], row["invest"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8,
    )

plt.title("1954年の企業市場価値と投資額")
plt.xlabel("市場価値")
plt.ylabel("投資額")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

## 9. 残差を企業別に確認する

残差は、実際の投資額から回帰式による予測値を引いた値である。

$$
\widehat u_i
=
invest_i
-
\widehat{invest}_i
$$

- 残差が正なら、実際の投資額は回帰式の予測より大きい。
- 残差が負なら、実際の投資額は回帰式の予測より小さい。
- 残差が大きい企業では、市場価値だけでは投資額を十分に説明できていない可能性がある。

In [ ]:
residual_table_1954 = (
    grunfeld_1954[
        ["firm", "invest", "value", "fitted", "residual"]
    ]
    .sort_values("residual")
)

residual_table_1954

### 演習3　残差を解釈する

次を確認しなさい。

1. 残差が最も大きい企業。
2. 残差が最も小さい企業。
3. 残差が正の企業について、モデルが投資額を過大予測しているか、過小予測しているか。
4. 市場価値以外に、企業の投資額と関係しそうな要因。
5. 市場価値以外の要因を考慮していないことが、残差の大きさにどのように関係するか。

In [ ]:
# ここに説明をコメントとして入力する


## 10. 1935年についても同じ単回帰を行う

1954年だけの関係が、別の年にも同じ形で見られるとは限らない。

そこで、データの最初の年である1935年を取り出し、同じ単回帰を行う。

$$
invest_i
=
\beta_0
+
\beta_1 value_i
+
u_i
$$

説明変数と被説明変数は変えない。年だけを変えて、傾きや決定係数がどのように異なるかを確認する。

In [ ]:
grunfeld_1935 = (
    grunfeld[grunfeld["year"] == 1935]
    .copy()
    .sort_values("value")
    .reset_index(drop=True)
)

Y_1935 = grunfeld_1935["invest"]
X_1935 = sm.add_constant(grunfeld_1935[["value"]])

result_1935 = sm.OLS(Y_1935, X_1935).fit()

year_comparison = pd.DataFrame({
    "1935年": {
        "切片": result_1935.params["const"],
        "市場価値の係数": result_1935.params["value"],
        "係数のp値": result_1935.pvalues["value"],
        "決定係数": result_1935.rsquared,
    },
    "1954年": {
        "切片": result_1954.params["const"],
        "市場価値の係数": result_1954.params["value"],
        "係数のp値": result_1954.pvalues["value"],
        "決定係数": result_1954.rsquared,
    },
})

year_comparison

## 11. 二つの年の回帰直線を比較する

1935年と1954年では、企業規模の分布も投資環境も異なる。二つの回帰直線を別々に描き、関係が時期によって変化しているかを確認する。

ここでは、1935年と1954年について別々に単回帰を推定し、係数と決定係数を記述的に比較する。

In [ ]:
grunfeld_1935["fitted"] = result_1935.fittedvalues

plt.figure(figsize=(8, 5))

plt.scatter(
    grunfeld_1935["value"],
    grunfeld_1935["invest"],
    marker="o",
    label="1935年",
)
plt.plot(
    grunfeld_1935["value"],
    grunfeld_1935["fitted"],
    label="1935年の回帰直線",
)

plt.scatter(
    grunfeld_1954["value"],
    grunfeld_1954["invest"],
    marker="x",
    label="1954年",
)
plt.plot(
    grunfeld_1954["value"],
    grunfeld_1954["fitted"],
    label="1954年の回帰直線",
)

plt.title("1935年と1954年の単回帰の比較")
plt.xlabel("市場価値")
plt.ylabel("投資額")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

## 13. パッケージ内データの利点と注意点

利点は、ファイルの配置を意識せず、短いコードで再現できることである。また、データの説明書がパッケージ内に含まれている場合が多い。

一方、次の点に注意する必要がある。

- パッケージに収録された時点で更新が止まっていることがある。
- 古い教科書データや海外データが多く、現在の日本経済を直接分析できるとは限らない。
- 元の公表機関が系列を改定しても、パッケージ内の値が自動的に更新されるとは限らない。
- 分析用に整理・簡略化されたデータである可能性がある。
- パッケージのバージョンによって内容が変わる可能性がある。

したがって、分析ではデータの出典、対象期間、取得方法、パッケージのバージョンを記録する。パッケージ内データは再現しやすい一方、政策分析や現状分析では、目的に合った最新の公的統計を別途取得する必要がある。

# 第2部　複数の公的統計CSVを結合する

分析する問いは、次のとおりである。

> 人口密度の高い都道府県ほど、住宅の床面積は小さいか。

使用するデータは、総務省統計局『統計でみる都道府県のすがた2026』の指標を、分析用のCSVへ整理したものである。

- 総面積1平方キロメートル当たり人口密度：2024年、指標コード `#A01201`。
- 可住地面積1平方キロメートル当たり人口密度：2024年、指標コード `#A01202`。
- 持ち家住宅の延べ面積、1住宅当たり：2023年、指標コード `#H0210301`。
- 借家住宅の延べ面積、1住宅当たり：2023年、指標コード `#H0210302`。

人口密度と住宅面積では基準年が1年異なる。この違いは分析上の注意事項として記録する。

## 14. 二つのCSVを読み込む

人口密度と住宅面積は別々のCSVに保存されている。

- `japan_prefecture_density_2024.csv`
- `japan_prefecture_housing_2023.csv`

最初に、それぞれを別のDataFrameとして読み込む。

In [ ]:
density = pd.read_csv(
    "japan_prefecture_density_2024.csv"
)

housing = pd.read_csv(
    "japan_prefecture_housing_2023.csv"
)

print("人口密度データ:", density.shape)
display(density.head())

print("住宅面積データ:", housing.shape)
display(housing.head())

## 16. `merge()` で一対一結合する

`pandas.merge()` を使い、二つの表を横方向に結合する。

`validate="one_to_one"` を指定すると、両方の表でキーが一意であることを確認できる。意図せず同じコードが複数行に存在する場合はエラーになる。

`indicator=True` を指定すると、各行がどちらの表から来たかを示す `_merge` 列が追加される。すべて `both` であれば、両方のCSVで対応する行が見つかったことを意味する。

In [ ]:
prefecture_data = density.merge(
    housing,
    on=["prefecture_code", "prefecture"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)

print(prefecture_data["_merge"].value_counts())
display(prefecture_data.head())

assert len(prefecture_data) == 47
assert prefecture_data["_merge"].eq("both").all()

prefecture_data = prefecture_data.drop(columns="_merge")

### 演習5　結合結果を検証する

次を確認しなさい。

1. 結合後の行数。
2. `_merge` がすべて `both` になっているか。
3. `validate="one_to_one"` が検査している条件。
4. 片方のCSVに同じ都道府県コードが2行あった場合、単純な結合で行数が増える理由。
5. 結合後に必要な列がすべて含まれているか。

In [ ]:
# ここにコードと説明を入力する
